<a href="https://www.kaggle.com/code/rudrajit55555/eda-rudrajit-das?scriptVersionId=359061468" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/nikhil1e9/loan-default/Loan_default.csv


# 1. Setup and Importing Libraries

In [2]:
import os, sys, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.inspection import permutation_importance
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_recall_curve, roc_curve,
                             recall_score, precision_score, fbeta_score, brier_score_loss, confusion_matrix)
from IPython.display import display

warnings.filterwarnings('ignore', category=FutureWarning)

random_state = 42
id_candidates = ['LoanID']

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)

print('Python:', sys.version.split()[0])
print('Working directory:', os.getcwd())

try:
    import statsmodels.api as sm
    HAS_SM = True
except ImportError:
    HAS_SM = False
try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False
print(f"statsmodels={HAS_SM}  lightgbm={HAS_LGB}  shap={HAS_SHAP}")

Python: 3.13.15
Working directory: /kaggle/working
statsmodels=True  lightgbm=True  shap=True


# 2. Loading the CSV data

In [3]:
df = pd.read_csv('/kaggle/input/datasets/nikhil1e9/loan-default/Loan_default.csv', low_memory=False)
cfg = dict(
    data_path=['/kaggle/input/datasets/nikhil1e9/loan-default/Loan_default.csv'],
    target='Default',
    target_recall=0.80,        # business constraint: catch >= 80% of defaulters
    cost_fn=5.0, cost_fp=1.0,  # relative cost of missing a default vs wrongly flagging a good loan (it is set with credit-policy team)
    test_size=0.20, val_size=0.20,
    n_search_iter=15, cv_folds=3,
    n_boot=300,
    protected=['MaritalStatus'],   # ECOA-sensitive: used for fairness audit; modeled with/without
    pricing_leak_suspect=['InterestRate'], # priced AFTER underwriting risk assessment -> potential leakage/circularity
)

In [4]:
# Normalize column names conservatively for robust matching, while retaining source names.
original_columns = df.columns.tolist()
normalized_lookup = {str(c).strip().lower(): c for c in df.columns}

# Reference the defined target from cfg and id_candidates list
target_candidates = [cfg['target']]

target_col = next(
    (normalized_lookup[c.lower()] for c in target_candidates if c.lower() in normalized_lookup),
    None
)
if target_col is None:
    print("Available columns:", original_columns)
    raise KeyError(
        "Could not identify the target. Set target_col manually to the column representing loan default."
    )

id_col = next(
    (normalized_lookup[c.lower()] for c in id_candidates if c.lower() in normalized_lookup),
    None
)

print("Target column:", target_col)
print("\nTarget values:")
display(df[target_col].value_counts(dropna=False).to_frame("count"))

Target column: Default

Target values:


,count
Default,
0,225694
1,29653


# 3. Data Cleaning

**3.1> Shape and Type**

In [5]:
print("Shape:", df.shape)
display(df.head())

Shape: (255347, 18)


,LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
0,I38PQUQS96,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0
1,HPSK72WA7R,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0
2,C1OZ6DPJ8Y,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1
3,V2KKSFM3UN,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0
4,EY08JDHTZP,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0


**3.2> Finding missing Values**

In [6]:
print("\nMissing values:")
print(df.isna().sum().sort_values(ascending=False))


Missing values:
LoanID            0
Age               0
Income            0
LoanAmount        0
CreditScore       0
MonthsEmployed    0
NumCreditLines    0
InterestRate      0
LoanTerm          0
DTIRatio          0
Education         0
EmploymentType    0
MaritalStatus     0
HasMortgage       0
HasDependents     0
LoanPurpose       0
HasCoSigner       0
Default           0
dtype: int64


**3.3> Searching for duplicates**

In [7]:
print("\nExact duplicate rows:", df.duplicated().sum())
print("\nDuplicate loan IDs:", df["LoanID"].duplicated().sum())


Exact duplicate rows: 0

Duplicate loan IDs: 0


**3.4> Range and Summary Sattistics**

In [8]:
print("\nRange Check & Summary Satistics:", df.describe())


Range Check & Summary Satistics:                  Age         Income     LoanAmount    CreditScore  MonthsEmployed  NumCreditLines   InterestRate       LoanTerm  \
count  255347.000000  255347.000000  255347.000000  255347.000000   255347.000000   255347.000000  255347.000000  255347.000000   
mean       43.498306   82499.304597  127578.865512     574.264346       59.541976        2.501036      13.492773      36.025894   
std        14.990258   38963.013729   70840.706142     158.903867       34.643376        1.117018       6.636443      16.969330   
min        18.000000   15000.000000    5000.000000     300.000000        0.000000        1.000000       2.000000      12.000000   
25%        31.000000   48825.500000   66156.000000     437.000000       30.000000        2.000000       7.770000      24.000000   
50%        43.000000   82466.000000  127556.000000     574.000000       60.000000        2.000000      13.460000      36.000000   
75%        56.000000  116219.000000  188985.00000

**3.5> Outliers for numerical columns**

In [9]:
bin = ["HasMortgage", "HasDependents", "HasCoSigner"]
for c in bin:
    if not pd.api.types.is_numeric_dtype(df[c]):
        df[c] = df[c].map({"Yes": 1, "No": 0})

def iqr_outlier_share(s):
    q1, q3 = s.quantile([.25, .75]); i = q3 - q1
    return ((s < q1 - 1.5*i) | (s > q3 + 1.5*i)).mean()

num_cols = ["Age","Income","LoanAmount","CreditScore","MonthsEmployed","NumCreditLines","InterestRate","LoanTerm","DTIRatio"]
display(pd.DataFrame({"skew": df[num_cols].skew(), "kurtosis": df[num_cols].kurtosis(),
                      "iqr_outlier_share": df[num_cols].apply(iqr_outlier_share)}).round(3))

def add_features(d):
    d = d.copy()
    d["LoanToIncome"] = d.LoanAmount / d.Income
    d["EMI"] = (d.LoanAmount * d.InterestRate * (1 + d.InterestRate)**d.LoanTerm) / ((1 + d.InterestRate)**d.LoanTerm - 1)   # compound-interest appropriation
    d["EMIToIncome"] = d.EMI / (d.Income/12)
    d["DebtBurden"] = d.DTIRatio * d.LoanToIncome
    d["TenureRatio"] = d.MonthsEmployed / ((d.Age - 18).clip(lower=1) * 12)
    d["AgeBand"] = pd.cut(d.Age, [17,25,35,50,100], labels=["18-25","26-35","36-50","51+"])
    d["IncomeQuartile"] = pd.qcut(d.Income, 4, labels=["Q1","Q2","Q3","Q4"])
    d["ScoreBand"] = pd.cut(d.CreditScore, [0,579,669,739,799,900], labels=["Poor","Fair","Good","VeryGood","Excellent"])
    d["RateBand"] = pd.cut(d.InterestRate, [0,8,13,18,100], labels=["<=8","8-13","13-18",">18"])
    d["TenureBand"] = pd.cut(d.MonthsEmployed, [-1,11,23,59,119,1000], labels=["<12","12-23","24-59","60-119","120+"])
    return d

dfe = add_features(df)
y = dfe[cfg["target"]].astype(int)

,skew,kurtosis,iqr_outlier_share
Age,0.001,-1.198,0.0
Income,-0.000,-1.198,0.0
LoanAmount,-0.002,-1.204,0.0
CreditScore,0.005,-1.200,0.0
MonthsEmployed,-0.002,-1.200,0.0
NumCreditLines,-0.000,-1.358,0.0
InterestRate,0.005,-1.197,0.0
LoanTerm,-0.002,-1.300,0.0
DTIRatio,-0.001,-1.200,0.0


**3.6> Category Values**

In [10]:
# Normalize common binary target representations to 0/1.
target_raw = df[target_col]
if target_raw.isna().any():
    raise ValueError("Target contains missing values. Investigate before modeling.")

if set(pd.Series(target_raw.dropna().unique()).tolist()).issubset({0, 1, 0.0, 1.0, True, False}):
    y_all = target_raw.astype(int)
else:
    target_map = {
        "0": 0, "1": 1,
        "no": 0, "yes": 1,
        "false": 0, "true": 1,
        "non-default": 0, "default": 1,
        "non_default": 0, "defaulted": 1,
    }
    mapped = target_raw.astype(str).str.strip().str.lower().map(target_map)
    if mapped.isna().any():
        print("Unrecognized target values:", target_raw.astype(str).value_counts().head(20))
        raise ValueError("Target is not recognized as binary. Define an explicit mapping.")
    y_all = mapped.astype(int)

print("Target prevalence:")
display(y_all.value_counts().rename(index={0: "Non-default (0)", 1: "Default (1)"}).to_frame("count"))
display((y_all.value_counts(normalize=True) * 100).rename(index={0: "Non-default (0)", 1: "Default (1)"}).round(2).to_frame("percent"))

if y_all.nunique() != 2:
    raise ValueError("Expected a binary target with classes 0 and 1.")

Target prevalence:


,count
Default,
Non-default (0),225694
Default (1),29653


,percent
Default,
Non-default (0),88.39
Default (1),11.61


In [11]:




print("\nTarget distribution:")
print(df["Default"].value_counts(dropna=False))

print("\nTarget proportions:")
print(df["Default"].value_counts(normalize=True, dropna=False))

print("\nNumeric summary:")
print(df.describe(include="all").T)


Target distribution:
Default
0    225694
1     29653
Name: count, dtype: int64

Target proportions:
Default
0    0.883872
1    0.116128
Name: proportion, dtype: float64

Numeric summary:
                   count  unique         top   freq           mean           std      min      25%       50%       75%       max
LoanID            255347  255347  ZTH91CGL0B      1            NaN           NaN      NaN      NaN       NaN       NaN       NaN
Age             255347.0     NaN         NaN    NaN      43.498306     14.990258     18.0     31.0      43.0      56.0      69.0
Income          255347.0     NaN         NaN    NaN   82499.304597  38963.013729  15000.0  48825.5   82466.0  116219.0  149999.0
LoanAmount      255347.0     NaN         NaN    NaN  127578.865512  70840.706142   5000.0  66156.0  127556.0  188985.0  249999.0
CreditScore     255347.0     NaN         NaN    NaN     574.264346    158.903867    300.0    437.0     574.0     712.0     849.0
MonthsEmployed  255347.0     NaN      